### Simple graphs
Implement your own graph class.
Let's start by defining a class for a very simple graph object:

In [1]:
class VerySimpleGraph:
    # Our graph has two attributes, a node list and an edge list
    def __init__(self):
        self.nodes = [] # list of node indices
        self.edges = [] # list of edge indices
    
    # we also define the string representation of our class, which allows us to use print()
    def __repr__(self): 
        repr_str = f"Graph size: # nodes: {len(self.nodes)}, # edges: {len(self.edges)}"
        repr_str += f"\nNode list: {self.nodes}"
        repr_str += f"\nEdge list: {self.edges}"
        return repr_str
    
    # next we need a function to add new nodes
    def add_node(self, node_id): 
        if node_id not in self.nodes:
            self.nodes.append(node_id)
    
    # and a function to add new edges
    def add_edge(self, src_id, dest_id):
        self.edges.append((src_id, dest_id))

Now, let's construct a very simple graph of 3 nodes connected by 2 edges

In [2]:
g = VerySimpleGraph()
g.add_node(1)
g.add_node(2)
g.add_node(3)
g.add_edge(1,2)
g.add_edge(1,3)
print(g)

Graph size: # nodes: 3, # edges: 2
Node list: [1, 2, 3]
Edge list: [(1, 2), (1, 3)]


Now, we want to add atributes to our nodes and edges (like IDs, names), so we create classes for nodes and edges

In [3]:
class Node:
    def __init__(self, node_id, node_name):
        self.id = node_id
        self.name = node_name
        
    def __repr__(self):
        return f'{self.id}: {self.name}'
      
class Edge:
    def __init__(self, edge_id, src_node_id, dest_node_id):
        self.id = edge_id
        self.source = src_node_id
        self.target = dest_node_id
    
    def __repr__(self):
        return f'{self.id}: ({self.source}, {self.target})'

class SimpleGraph:
    def __init__(self):
        self.nodes = {}
        self.edges = []
        
    def __repr__(self):
        repr_str = f"Graph size: # nodes: {len(self.nodes)}, # edges: {len(self.edges)}"
        repr_str += f"\nNode list: {self.nodes}"
        repr_str += f"\nEdge list: {self.edges}"
        return repr_str
    
    def add_node(self, node_object):
        self.nodes[node_object.id] = node_object
        
    def add_edge(self, edge_object):
        self.edges.append(edge_object)


In [4]:
g = SimpleGraph()

# We now have to create node objects
node1 = Node(1, 'node1')
node2 = Node(2, 'node2')
node3 = Node(3, 'node3')

# Add the node objects to the graph
g.add_node(node1)
g.add_node(node2)
g.add_node(node3)

# Create edge objects
edge1 = Edge(1, node1.id, node2.id)
edge2 = Edge(2, node2.id, node3.id)

# Add edge objects to the graph
g.add_edge(edge1)
g.add_edge(edge2)
print(g)

Graph size: # nodes: 3, # edges: 2
Node list: {1: 1: node1, 2: 2: node2, 3: 3: node3}
Edge list: [1: (1, 2), 2: (2, 3)]


Now, let's get fancy and create a graph object that can represent a molecule

In [5]:
# First, we define a helper function to translate edge weights into characters representing the number of bonds
def weight_to_char(weight):
    if weight == 1:
        return "-"
    elif weight == 2:
        return "="
    elif weight == 3:
        return "#"
    else:
        return "?"

In [6]:
# Node, Edge, and MolecularGraph classes
import numpy as np

class Node:
    def __init__(self, atom_type):
        self.id = None
        self.atom = atom_type
        
    def __repr__(self):
        return f'{self.id}: {self.atom}'
      
class Edge:
    def __init__(self, source_node_id, dest_node_id, weight=1):
        self.id = None
        self.source = source_node_id
        self.target = dest_node_id
        self.weight = weight
    
    def __repr__(self):
        return f'{self.id}: ({self.source}, {self.target}, {self.weight})'

class MoleculeGraph:
    def __init__(self):
        self.nodes = {} # Dictionary of node (atom) information
        self.edges = [] # Edge list
        
    def __repr__(self):
        repr_str = f"Graph size: # nodes: {len(self.nodes)}, # edges: {len(self.edges)}"
        repr_str += f"\nNode list: {self.nodes}"
        repr_str += f"\nEdge list: {self.edges}"
        repr_str += f"\nMolecule: {self.draw_molecule()}"
        repr_str += f'\nAdjecency matrix:\n'
        repr_str += str(self.get_adjacency_matrix())
        return repr_str
    
    def add_node(self, node_object):
        new_id = len(self.nodes) # create new index
        node_object.id = new_id
        self.nodes[new_id] = node_object
    
    def add_nodes(self, list_of_nodes):
        for node in list_of_nodes:
            self.add_node(node)
        
    def add_edge(self, edge_object):
        assert edge_object.source in self.nodes.keys(), f"Source node with ID {edge_object.source } is not present in graph"
        assert edge_object.target in self.nodes.keys(), f"Target node with ID {edge_object.target } is not present in graph"
        edge_object.id = len(self.edges)
        self.edges.append(edge_object)
    
    def get_next_edge(self, node_id, already_seen):
        for edge in self.edges:
            if edge.source == node_id and edge.id not in already_seen:
                return edge
    
    def draw_molecule(self):
        current_edge = self.edges[0] 
        output_string = self.nodes[current_edge.source].atom # the first atom of the first edge
        already_seen = []
        while len(already_seen) != len(self.edges):
            output_string += weight_to_char(current_edge.weight) # add bonds
            output_string += self.nodes[current_edge.target].atom # add target atom
            
            already_seen.append(current_edge.id) # expand the list fo nodes that we have already added
            current_edge = self.get_next_edge(current_edge.target, already_seen) # define new current edge
        return output_string

    def get_adjacency_matrix(self):
        m = np.zeros((len(self.nodes),len(self.nodes)), dtype=int) # create 0 matrix n_nodes x n_nodes
        for edge in self.edges:
            m[edge.source][edge.target] = edge.weight
            m[edge.target][edge.source] = edge.weight
        return m

In [ ]:
g = MoleculeGraph()
carbon1 = Node('C')
carbon2 = Node('C')
oxygen = Node('O')

g.add_nodes([carbon1, carbon2, oxygen])

single_bond_C1_C2 = Edge(carbon1.id, carbon2.id, weight=1)
double_bond_C2_O = Edge(carbon2.id, oxygen.id, weight=2)

g.add_edge(single_bond_C1_C2)
g.add_edge(double_bond_C2_O)
print(g)
g.get_adjacency_matrix()

Graph size: # nodes: 3, # edges: 2
Node list: {0: 0: C, 1: 1: C, 2: 2: O}
Edge list: [0: (0, 1, 1), 1: (1, 2, 2)]
Molecule: C-C=O
Adjecency matrix:
[[0 1 0]
 [1 0 2]
 [0 2 0]]


array([[0, 1, 0],
       [1, 0, 2],
       [0, 2, 0]])

: 

### Questions:
- Do we have a directed or an undirected graph here?
- Can you define a graph for a more complex molecule? 
- What are the challenges? What is missing from the MoleculeGraph?
- Can you define a function that finds all aldehydes in a molecule?
- Can you define a function that reduces all aldehydes to alcohol groups?
